# Silent bystander pegRNA design — CH / AML variants

Testing `pegg` (this working copy, not an installed version) on clonal
haematopoiesis and AML mutations of interest.

**Kernel:** use the `pegg_env` environment — the Azimuth and RandomForest
scorers are pickled against scikit-learn 1.1.1 and fail on newer versions.

## Import the local pegg

In [1]:
import sys, os

# import THIS working copy of pegg, not any pip-installed version.
# the notebook lives in tests/, so the package root is one level up.
REPO = os.path.abspath('..')
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import importlib
import pegg
from pegg import prime, base, library, bystander

# reload after editing the source, so the notebook picks up changes
for m in (bystander, library, base, prime):
    importlib.reload(m)

print('pegg loaded from:', os.path.dirname(pegg.__file__))
print('silent_bystander supported:', 'silent_bystander' in prime.run.__code__.co_varnames)

/opt/miniconda3/envs/pegg_env/lib/python3.9/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


pegg loaded from: /Users/kexindong/Documents/GitHub/PEGG3.0/pegg
silent_bystander supported: True


In [2]:
import pandas as pd
import numpy as np
import Bio.Seq

print('python  :', sys.version.split()[0])
import sklearn; print('sklearn :', sklearn.__version__, '(needs 1.1.1 for the scorers)')
print('pandas  :', pd.__version__)

python  : 3.9.23
sklearn : 1.1.1 (needs 1.1.1 for the scorers)
pandas  : 2.3.1


## Input variants from clinical data

cBioPortal format. Required columns:
`Hugo_Symbol, Chromosome, Start_Position, End_Position, Variant_Type,
Reference_Allele, Tumor_Seq_Allele2`

Replace this with your CH / AML variant table.

In [17]:
df_mutation = pd.read_csv('/Users/kexindong/Documents/GitHub/Database/PublicDatabase/AACR-GENIE/v17.0/data_mutations_extended.txt', header=0, sep='\t', comment="#", na_values = 'Not Applicable')

In [18]:
df_mutation

,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Consequence,Variant_Classification,...,FILTER,Polyphen_Prediction,Polyphen_Score,SIFT_Prediction,SIFT_Score,SWISSPROT,n_depth,t_depth,Annotation_Status,mutationInCis_Flag
0,KRAS,3845.0,JHU,GRCh37,12,25398285,25398285,+,missense_variant,Missense_Mutation,...,PASS,probably_damaging,0.991,deleterious,0.04,NaN,NaN,1623.0,SUCCESS,False
1,BRAF,673.0,JHU,GRCh37,7,140453136,140453136,+,missense_variant,Missense_Mutation,...,PASS,probably_damaging,0.963,deleterious,0.00,NaN,NaN,1031.0,SUCCESS,False
2,EGFR,1956.0,JHU,GRCh37,7,55249071,55249071,+,missense_variant,Missense_Mutation,...,PASS,probably_damaging,1.000,deleterious,0.00,NaN,NaN,692.0,SUCCESS,False
3,TP53,7157.0,JHU,GRCh37,17,7577120,7577120,+,missense_variant,Missense_Mutation,...,PASS,possibly_damaging,0.643,tolerated,0.13,NaN,NaN,930.0,SUCCESS,False
4,NRAS,4893.0,JHU,GRCh37,1,115256529,115256529,+,missense_variant,Missense_Mutation,...,PASS,benign,0.251,tolerated,0.06,NaN,NaN,2277.0,SUCCESS,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2266024,CBL,867.0,PROV,GRCh37,11,119148875,119148875,+,splice_acceptor_variant,Splice_Site,...,PASS,NaN,NaN,NaN,NaN,NaN,NaN,423.0,SUCCESS,False
2266025,AR,367.0,PROV,GRCh37,X,66765014,66765014,+,missense_variant,Missense_Mutation,...,PASS,probably_damaging,0.990,deleterious_low_confidence,0.01,NaN,NaN,5517.0,SUCCESS,False
2266026,MSH3,4437.0,PROV,GRCh37,5,80021325,80021325,+,missense_variant,Missense_Mutation,...,PASS,probably_damaging,0.934,deleterious,0.01,NaN,NaN,510.0,SUCCESS,False
2266027,ATM,472.0,PROV,GRCh37,11,108098589,108098589,+,missense_variant,Missense_Mutation,...,PASS,benign,0.041,tolerated,0.35,NaN,NaN,4161.0,SUCCESS,False


In [19]:
CH = ['DNMT3A','TET2','ASXL1','IDH1','IDH2','SF3B1','SRSF2','U2AF1','JAK2','TP53','PPM1D',]
AML = ['FLT3','NRAS','KRAS','KIT','PTPN11','NPM1','RUNX1','WT1']
HSC = CH + AML

In [20]:
# keep mutation in the genes of interest
df_mutation = df_mutation[df_mutation['Hugo_Symbol'].isin(HSC)].reset_index(drop=True)
df_mutation

,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Consequence,Variant_Classification,...,FILTER,Polyphen_Prediction,Polyphen_Score,SIFT_Prediction,SIFT_Score,SWISSPROT,n_depth,t_depth,Annotation_Status,mutationInCis_Flag
0,KRAS,3845.0,JHU,GRCh37,12,25398285,25398285,+,missense_variant,Missense_Mutation,...,PASS,probably_damaging,0.991,deleterious,0.04,NaN,NaN,1623.0,SUCCESS,False
1,TP53,7157.0,JHU,GRCh37,17,7577120,7577120,+,missense_variant,Missense_Mutation,...,PASS,possibly_damaging,0.643,tolerated,0.13,NaN,NaN,930.0,SUCCESS,False
2,NRAS,4893.0,JHU,GRCh37,1,115256529,115256529,+,missense_variant,Missense_Mutation,...,PASS,benign,0.251,tolerated,0.06,NaN,NaN,2277.0,SUCCESS,False
3,KIT,3815.0,JHU,GRCh37,4,55593673,55593673,+,missense_variant,Missense_Mutation,...,PASS,benign,0.177,tolerated,0.07,NaN,NaN,2015.0,SUCCESS,False
4,KRAS,3845.0,JHU,GRCh37,12,25378636,25378636,+,missense_variant,Missense_Mutation,...,PASS,possibly_damaging,0.797,deleterious,0.03,NaN,NaN,1210.0,SUCCESS,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245370,TET2,54790.0,PROV,GRCh37,4,106157755,106157755,+,stop_gained,Nonsense_Mutation,...,PASS,NaN,NaN,NaN,NaN,NaN,NaN,796.0,SUCCESS,False
245371,TP53,7157.0,PROV,GRCh37,17,7577120,7577120,+,missense_variant,Missense_Mutation,...,PASS,possibly_damaging,0.643,tolerated,0.13,NaN,NaN,5174.0,SUCCESS,False
245372,TP53,7157.0,PROV,GRCh37,17,7579312,7579312,+,"splice_region_variant,synonymous_variant",Splice_Region,...,PASS,NaN,NaN,NaN,NaN,NaN,NaN,3352.0,SUCCESS,False
245373,TET2,54790.0,PROV,GRCh37,4,106157573,106157573,+,stop_gained,Nonsense_Mutation,...,PASS,NaN,NaN,NaN,NaN,NaN,NaN,643.0,SUCCESS,False


In [ ]:
# # drop duplicates based on the columns that define a unique mutation
# df_mutation = df_mutation.drop_duplicates(subset=['Hugo_Symbol', 'Chromosome', 'Start_Position', 'End_Position', 'Reference_Allele', 'Tumor_Seq_Allele2','Variant_Type','Variant_Classification']).reset_index(drop=True)
# df_mutation

,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Consequence,Variant_Classification,...,FILTER,Polyphen_Prediction,Polyphen_Score,SIFT_Prediction,SIFT_Score,SWISSPROT,n_depth,t_depth,Annotation_Status,mutationInCis_Flag
0,KRAS,3845.0,JHU,GRCh37,12,25398285,25398285,+,missense_variant,Missense_Mutation,...,PASS,probably_damaging,0.991,deleterious,0.04,NaN,NaN,1623.0,SUCCESS,False
1,TP53,7157.0,JHU,GRCh37,17,7577120,7577120,+,missense_variant,Missense_Mutation,...,PASS,possibly_damaging,0.643,tolerated,0.13,NaN,NaN,930.0,SUCCESS,False
2,NRAS,4893.0,JHU,GRCh37,1,115256529,115256529,+,missense_variant,Missense_Mutation,...,PASS,benign,0.251,tolerated,0.06,NaN,NaN,2277.0,SUCCESS,False
3,KIT,3815.0,JHU,GRCh37,4,55593673,55593673,+,missense_variant,Missense_Mutation,...,PASS,benign,0.177,tolerated,0.07,NaN,NaN,2015.0,SUCCESS,False
4,KRAS,3845.0,JHU,GRCh37,12,25378636,25378636,+,missense_variant,Missense_Mutation,...,PASS,possibly_damaging,0.797,deleterious,0.03,NaN,NaN,1210.0,SUCCESS,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43517,TP53,7157.0,PROV,GRCh37,17,7579398,7579398,+,missense_variant,Missense_Mutation,...,PASS,probably_damaging,0.998,deleterious,0.04,NaN,NaN,5140.0,SUCCESS,False
43518,DNMT3A,1788.0,PROV,GRCh37,2,25457289,25457289,+,"missense_variant,splice_region_variant",Missense_Mutation,...,PASS,probably_damaging,0.989,deleterious,0.01,NaN,NaN,1413.0,SUCCESS,False
43519,TET2,54790.0,PROV,GRCh37,4,106196511,106196512,+,frameshift_variant,Frame_Shift_Ins,...,PASS,NaN,NaN,NaN,NaN,NaN,NaN,992.0,SUCCESS,False
43520,TET2,54790.0,PROV,GRCh37,4,106157407,106157407,+,stop_gained,Nonsense_Mutation,...,PASS,NaN,NaN,NaN,NaN,NaN,NaN,671.0,SUCCESS,False


In [21]:
VARIANT_KEY = ['Chromosome', 'Start_Position', 'End_Position',
               'Reference_Allele', 'Tumor_Seq_Allele2']

LOF_CLASSES = ['Nonsense_Mutation', 'Frame_Shift_Del',
               'Frame_Shift_Ins', 'Splice_Site']


def filter_variants(df, classifications=None, protein_change=None,
                    key=None, count_col='count', per_sample=True,
                    sample_col='Tumor_Sample_Barcode'):
    """
    Filters a MAF-style mutation table and collapses recurrent variants into one
    row each, carrying a count of how often the variant was observed.

    Parameters
    -----------
    df
        *type = pd.DataFrame*

        MAF-style table, e.g. one gene's rows pulled from cBioPortal.

    classifications
        *type = list or None*

        Variant_Classification values to keep. Default = None (no filter).

    protein_change
        *type = str, list or None*

        HGVSp_Short value(s) to keep, e.g. 'p.R882H'. Default = None.

    key
        *type = list or None*

        Columns defining variant identity. Default = None (VARIANT_KEY).

    count_col
        *type = str*

        Name of the appended count column. Default = 'count'.

    per_sample
        *type = bool*

        If True, count distinct samples rather than rows, so that a sample
        sequenced twice does not inflate the count. Default = True.

    sample_col
        *type = str*

        Column holding the sample identifier. Default = 'Tumor_Sample_Barcode'.
    """
    out = df

    if classifications is not None:
        out = out[out['Variant_Classification'].isin(classifications)]

    if protein_change is not None:
        wanted = [protein_change] if isinstance(protein_change, str) \
            else protein_change
        out = out[out['HGVSp_Short'].isin(wanted)]

    key = [k for k in (key or VARIANT_KEY) if k in out.columns]
    if len(key) == 0:
        raise ValueError('none of the key columns are present in df')

    if len(out) == 0:
        return out.assign(**{count_col: pd.Series(dtype=int)}).reset_index(drop=True)

    if per_sample and sample_col in out.columns:
        counts = (out.groupby(key, dropna=False)[sample_col]
                  .nunique().rename(count_col).reset_index())
    else:
        counts = (out.groupby(key, dropna=False)
                  .size().rename(count_col).reset_index())

    out = out.drop_duplicates(subset=key).merge(counts, on=key, how='left')

    return out.sort_values(count_col, ascending=False).reset_index(drop=True)

## DNMT3A

In [23]:
# for each gene, filter to the most common mutations

# DNMT3A
DNMT3A = df_mutation[df_mutation['Hugo_Symbol'] == 'DNMT3A'].reset_index(drop=True)
DNMT3A_LOF = filter_variants(DNMT3A, classifications=LOF_CLASSES)
DNMT3A_R882H = filter_variants(DNMT3A, protein_change=['p.R882H','R882H'])

In [25]:
DNMT3A_LOF

,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Consequence,Variant_Classification,...,Polyphen_Prediction,Polyphen_Score,SIFT_Prediction,SIFT_Score,SWISSPROT,n_depth,t_depth,Annotation_Status,mutationInCis_Flag,count
0,DNMT3A,1788.0,DFCI,GRCh37,2,25463182,25463182,+,stop_gained,Nonsense_Mutation,...,NaN,NaN,NaN,NaN,NaN,NaN,131.0,SUCCESS,False,106
1,DNMT3A,1788.0,DFCI,GRCh37,2,25467083,25467083,+,stop_gained,Nonsense_Mutation,...,NaN,NaN,NaN,NaN,NaN,NaN,304.0,SUCCESS,False,43
2,DNMT3A,1788.0,DFCI,GRCh37,2,25470516,25470516,+,stop_gained,Nonsense_Mutation,...,NaN,NaN,NaN,NaN,NaN,NaN,387.0,SUCCESS,False,34
3,DNMT3A,1788.0,DFCI,GRCh37,2,25523009,25523009,+,"frameshift_variant,splice_region_variant",Frame_Shift_Del,...,NaN,NaN,NaN,NaN,NaN,NaN,333.0,SUCCESS,False,33
4,DNMT3A,1788.0,DFCI,GRCh37,2,25470484,25470484,+,stop_gained,Nonsense_Mutation,...,NaN,NaN,NaN,NaN,NaN,NaN,360.0,SUCCESS,False,25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1092,DNMT3A,1788.0,MSK,GRCh37,2,25469514,25469515,+,frameshift_variant,Frame_Shift_Ins,...,NaN,NaN,NaN,NaN,NaN,817.0,755.0,SUCCESS,False,1
1093,DNMT3A,1788.0,MSK,GRCh37,2,25467414,25467414,+,frameshift_variant,Frame_Shift_Del,...,NaN,NaN,NaN,NaN,NaN,480.0,490.0,SUCCESS,False,1
1094,DNMT3A,1788.0,MSK,GRCh37,2,25469945,25469946,+,frameshift_variant,Frame_Shift_Ins,...,NaN,NaN,NaN,NaN,NaN,410.0,453.0,SUCCESS,False,1
1095,DNMT3A,1788.0,MSK,GRCh37,2,25505502,25505502,+,stop_gained,Nonsense_Mutation,...,NaN,NaN,NaN,NaN,NaN,518.0,566.0,SUCCESS,False,1


In [26]:
DNMT3A_R882H

,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Consequence,Variant_Classification,...,Polyphen_Prediction,Polyphen_Score,SIFT_Prediction,SIFT_Score,SWISSPROT,n_depth,t_depth,Annotation_Status,mutationInCis_Flag,count
0,DNMT3A,1788.0,DFCI,GRCh37,2,25457242,25457242,+,missense_variant,Missense_Mutation,...,benign,0.067,deleterious,0.0,NaN,NaN,422.0,SUCCESS,False,876


In [ ]:
DNMT3A_LOF_SELECTED = DNMT3A_LOF[DNMT3A_LOF['HGVSp_Short'].isin(['p.R771*', 'p.R598*','p.R320*'])].reset_index(drop=True)
DNMT3A_LOF_SELECTED

,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Consequence,Variant_Classification,...,Polyphen_Prediction,Polyphen_Score,SIFT_Prediction,SIFT_Score,SWISSPROT,n_depth,t_depth,Annotation_Status,mutationInCis_Flag,count
0,DNMT3A,1788.0,DFCI,GRCh37,2,25463182,25463182,+,stop_gained,Nonsense_Mutation,...,NaN,NaN,NaN,NaN,NaN,NaN,131.0,SUCCESS,False,106
1,DNMT3A,1788.0,DFCI,GRCh37,2,25467083,25467083,+,stop_gained,Nonsense_Mutation,...,NaN,NaN,NaN,NaN,NaN,NaN,304.0,SUCCESS,False,43
2,DNMT3A,1788.0,DFCI,GRCh37,2,25470516,25470516,+,stop_gained,Nonsense_Mutation,...,NaN,NaN,NaN,NaN,NaN,NaN,387.0,SUCCESS,False,34
3,DNMT3A,1788.0,MSK,GRCh37,2,25470516,25470517,+,stop_gained,Nonsense_Mutation,...,NaN,NaN,NaN,NaN,NaN,491.0,674.0,SUCCESS,False,1


In [ ]:
DNMT3A_SELECTED = DNMT3A_R882H.append(DNMT3A_LOF_SELECTED).reset_index(drop=True)

## TP53

## Load the reference genome

GRCh37 is the build the usual CDS coordinate sets are given in. This takes
~20 s and a few GB of RAM.

In [ ]:
# --- placeholder: swap in your own variant table ---
# mutations = pd.read_csv('path/to/your_ch_aml_variants.csv')

mutations = pd.DataFrame([
    # DNMT3A R882H (GRCh37 chr2) - the classic CH driver
    dict(Hugo_Symbol='DNMT3A', Chromosome=2, Start_Position=25457242,
         End_Position=25457242, Variant_Type='SNP',
         Reference_Allele='C', Tumor_Seq_Allele2='T'),
])

mutations

## Load the reference genome

GRCh37 is the build the usual CDS coordinate sets are given in. This takes
~20 s and a few GB of RAM.

## Reading frame annotation

Silent bystanders need the reading frame. For cBioPortal input that means the
transcript's CDS blocks (1-based, inclusive, ordered along the **+** strand)
plus the strand the transcript is on.

The annotation is attached to the variant table, so `run()` reads the frame per
row and a single call covers every gene.


In [ ]:
# --- CDS annotation for the genes being designed ---------------------------
# The reading frame is what makes a bystander "silent", so it has to come from a
# real transcript. Pull it from a GTF/GFF3 rather than typing coordinates by hand.
#
# GRCh37 annotation to match the genome loaded above, e.g.
#   GENCODE : gencode.v19.annotation.gtf.gz
#   Ensembl : Homo_sapiens.GRCh37.87.gtf.gz
GTF = '/Users/kexindong/Documents/GitHub/Database/RefGenome/gencode.v19.annotation.gtf.gz'

design_input = DNMT3A_SELECTED          # <- the variants being designed against

GENE_CDS = bystander.cds_from_gtf(
    GTF,
    genes=list(design_input['Hugo_Symbol'].unique()),
    # DNMT3A canonical transcript; drop this to take the longest CDS per gene
    transcript_ids={'DNMT3A': 'ENST00000264709'},
)

# Check it BEFORE designing. This catches: a CDS that is not a whole number of
# codons (wrong/partial transcript), overlapping blocks, genes with no annotation,
# and variants that fall outside their own gene's CDS -- all of which silently
# produce zero bystanders otherwise.
# Attach the annotation to the variant table. prime.run() reads the reading frame
# from these columns per row, so one call can span every gene in the library.
design_input = bystander.attach_cds(design_input, GENE_CDS)

report = bystander.cds_annotation_report(GENE_CDS, mutations=design_input)
report

## Design pegRNAs

In [ ]:
# --- design pegRNAs: one call for the whole library ------------------------
# The reading frame travels with each row (attached above), so genes with
# different transcripts -- and genes with no annotation at all -- can share a
# single run() call. Unannotated variants simply get ordinary pegRNAs.

peg_df = prime.run(
    design_input, 'cBioPortal',
    chrom_dict=chrom_dict,
    PAM='NGG',
    RTT_lengths=[10, 15, 20, 25, 30],
    PBS_lengths=[10, 13, 15],
    pegRNAs_per_mut=10,          # applies to EACH design type when bystanders are on
    min_RHA_size=1,
    RE_sites=['CGTCTC'],         # Esp3I / BsmBI
    sensor=True,
    sensor_length=60,
    before_proto_context=5,
    silent_bystander=True,
    silent_per_mut=2,            # bystander designs per pegRNA
    seed=0,                      # reproducible library
)

# standard library filters
peg_df = peg_df[peg_df['contains_polyT_terminator'] == False]
peg_df = peg_df[peg_df['sensor_error'] == 'No Error'].reset_index(drop=True)

peg_df = prime.prime_oligo_generator(peg_df)

print('rows: %d  (plain %d, bystander %d)' % (
    len(peg_df),
    (~peg_df['has_silent_bystander']).sum(),
    peg_df['has_silent_bystander'].sum()))

# per-variant breakdown -- every variant should retain BOTH design types
peg_df.groupby(['Hugo_Symbol', 'HGVSp_Short' if 'HGVSp_Short' in peg_df.columns
                else 'Start_Position', 'has_silent_bystander']).size().unstack(fill_value=0)

## Inspect

`has_silent_bystander` separates the two design types; both are present so
either can be filtered out.

In [ ]:
cols = ['Hugo_Symbol', 'Start_Position', 'pegRNA_id', 'Protospacer', 'RTT', 'PBS',
        'has_silent_bystander', 'n_bystander_muts', 'bystander_positions',
        'bystander_dist_to_edit', 'PAM_disrupted', 'PAM_disrupted_edit',
        'PAM_disrupted_by_bystander', 'PEGG2_Score', 'RF_Score',
        'pegRNA_rank_within_group']

peg_df[cols].head(20)

In [ ]:
# a bystander design next to the plain design it came from
g = peg_df.groupby(['mutation_idx', 'PAM_start', 'PAM_strand',
                    'RTT_length', 'PBS_length'])

for _, grp in g:
    if grp['has_silent_bystander'].any() and (~grp['has_silent_bystander']).any():
        plain = grp[~grp['has_silent_bystander']].iloc[0]
        print('plain     RTT =', plain['RTT'])
        print('          sensor_alt =', plain['sensor_alt'])
        for i in range(grp['has_silent_bystander'].sum()):
            k = grp[grp['has_silent_bystander']].iloc[i]
            diff = ''.join('.' if a == b else b.lower()
                           for a, b in zip(plain['RTT'], k['RTT']))
            print('bystander RTT =', k['RTT'], ' diff =', diff,
                  '(%d silent muts)' % k['n_bystander_muts'])
        break

## Check the bystanders really are silent

Applies each RTT back to the chromosome and re-translates the whole CDS. Every
pegRNA should change **at most one** amino acid — the intended edit. Two or more
would mean a bystander was not synonymous.

In [ ]:
# --- verify the bystanders really are silent -------------------------------
# Writes each pegRNA's RTT back onto the chromosome, re-translates the whole CDS,
# and counts how many amino acids changed. Every pegRNA should change AT MOST ONE
# (the intended edit). Two or more means a bystander was not synonymous.

from collections import Counter


def translate_cds(chrom_seq, cds_blocks, strand):
    pos = bystander.cds_positions(cds_blocks, strand)
    if strand == '-':
        bases = [str(Bio.Seq.Seq(str(chrom_seq[p - 1])).complement()) for p in pos]
    else:
        bases = [str(chrom_seq[p - 1]) for p in pos]
    return str(Bio.Seq.Seq(''.join(bases).upper()).translate())


def protein_after_edit(row, chrom_seq, cds_blocks, strand):
    """Apply this pegRNA's RTT to the genome and translate the resulting CDS."""
    rtt = str(Bio.Seq.Seq(row['RTT']).reverse_complement())   # stored revcomp'd
    if row['PAM_strand'] == '-':
        rtt = str(Bio.Seq.Seq(rtt).reverse_complement())

    alt_len = 0 if pd.isna(row['ALT']) else len(str(row['ALT']))
    d, rha = int(row['Distance_to_nick']), int(row['RHA_size'])
    edit_g = int(row['seq_start']) + len(str(row['left_context']))
    g0 = edit_g - d if row['PAM_strand'] == '+' else edit_g - rha
    if len(rtt) != d + alt_len + rha:
        return None

    edited = list(str(chrom_seq))
    for i, ch in enumerate(rtt):
        edited[g0 - 1 + i] = ch
    return translate_cds(''.join(edited), cds_blocks, strand)


for gene, sub in peg_df.groupby('Hugo_Symbol'):
    ann = GENE_CDS.get(gene)
    if ann is None or not ann['valid']:
        continue

    chrom_key = sub.iloc[0]['Chromosome']
    ref_prot = translate_cds(str(chrom_dict[chrom_key]).upper(),
                             ann['cds'], ann['strand'])

    counts = {}
    for _, r in sub.iterrows():
        p = protein_after_edit(r, str(chrom_dict[chrom_key]).upper(),
                               ann['cds'], ann['strand'])
        if p is None:
            continue
        n = sum(1 for a, b in zip(ref_prot, p) if a != b)
        counts.setdefault(bool(r['has_silent_bystander']), []).append(n)

    worst = max((n for v in counts.values() for n in v), default=0)
    print('%s  reference protein %d aa' % (gene, len(ref_prot)))
    for flag in (False, True):
        if flag in counts:
            print('   has_silent_bystander=%-5s aa changes: %s'
                  % (flag, dict(sorted(Counter(counts[flag]).items()))))
    print('   MAX amino-acid changes: %d   %s\n'
          % (worst, 'OK' if worst <= 1 else '*** A BYSTANDER IS NOT SILENT ***'))

## Export

In [ ]:
peg_df.to_csv('ch_aml_pegRNAs.csv', index=False)
print('wrote ch_aml_pegRNAs.csv  (%d rows, %d columns)' % peg_df.shape)

# oligos only
peg_df[['pegRNA_id', 'pegRNA_oligo']].to_csv('ch_aml_oligos.csv', index=False)
print('wrote ch_aml_oligos.csv')